#### EDA-Feature Engineering pipeline
Для меня самое главное - понять как гипотезы появляются в процессе:

"Раскопки данных" -> Построение гипотез  по тому как устроены данные и репрезентативных графиков в доказательство своих мылсей о данных -> Осмысленные выводы и гипотезы/предложения по созданию features ->  обучение BL моделей с feature engineering и без для сравнения полученного эффекта

# 1. Первичный анализ данных

На этом этапе выполняется первичный осмотр датасета без построения гипотез и без анализа зависимостей между признаками и стоимостью дома.

Задачи этапа:

- определить размер датасета;
- проверить строки и столбцы;
- определить целевую переменную;
- посмотреть распределение целевой переменной;
- изучить типы и смысл признаков;
- проверить количество уникальных значений;
- найти технические пропуски;
- отделить пропуски от значений, означающих отсутствие объекта;
- проверить дубликаты и базовую согласованность данных.

## 1. Импорты и загрузка данных

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


DATA_PATH = Path("data_raw/train.csv")
DESCRIPTION_PATH = Path("data_raw/data_description.txt")

TARGET = "SalePrice"
ID_COLUMN = "Id"

### Особенность загрузки данных

В исходном CSV используются строковые значения:

- `NA` — иногда отсутствие объекта, например отсутствие гаража или бассейна или ПРОПУСК;
- `None` — полноценная допустимая категория признака `MasVnrType`, означающая отсутствие каменной облицовки. .
- `NaN` - пропущенное значение в Pandas.

Стандартный вызов `pd.read_csv()` может автоматически преобразовать и `NA`, и `None` в `NaN`.

Поэтому сначала загрузим файл без автоматического распознавания пропусков, а затем отдельно преобразуем необходимые числовые столбцы.

In [ ]:
# raw_df сохраняет исходные значения CSV без автоматической
# замены строк "NA" и "None" на NaN.
raw_df = pd.read_csv(
    DATA_PATH,
    keep_default_na=False,
)

df = raw_df.copy()


# В этих числовых столбцах строка "NA" должна быть преобразована в NaN.
numeric_columns_with_na = [
    "LotFrontage",
    "MasVnrArea",
    "GarageYrBlt",
]

for column in numeric_columns_with_na:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce", # Если значение нельзя преобразовать в число, оно заменяется на NaN.
    )


# В Electrical есть одна строка "NA".
# В отличие от признаков вроде GarageType, здесь "NA"
# не является отдельной категорией из data_description.
df["Electrical"] = df["Electrical"].replace(
    "NA",
    pd.NA,
)

## 2. Общая информация о датасете

In [3]:
dataset_overview = pd.Series(
    {
        "Количество строк": df.shape[0],
        "Количество столбцов": df.shape[1],
        "Количество признаков без Id": (
            df.shape[1] - 2
        ),
        "Количество числовых столбцов": (
            df.select_dtypes(include=np.number).shape[1]
        ),
        "Количество object-столбцов": (
            df.select_dtypes(include="object").shape[1]
        ),
        "Количество дубликатов строк": (
            df.duplicated().sum()
        ),
        "Количество уникальных Id": (
            df[ID_COLUMN].nunique()
        ),
        "Количество пропусков в таргете": (
            df[TARGET].isna().sum()
        ),
    },
    name="value",
)

display(dataset_overview.to_frame())

,value
Количество строк,1460
Количество столбцов,81
Количество признаков без Id,79
Количество числовых столбцов,38
Количество object-столбцов,43
Количество дубликатов строк,0
Количество уникальных Id,1460
Количество пропусков в таргете,0


### Общая структура датасета

В датасете находится 1460 наблюдений и 81 столбец:

- `Id` — идентификатор объекта;
- 79 признаков, описывающих дом, участок и условия продажи;
- `SalePrice` — целевая переменная.

Все 1460 значений `Id` уникальны.

Полных дубликатов строк нет.

В целевой переменной `SalePrice` пропусков нет.

## 3. Просмотр строк датасета

In [4]:
display(df.head())

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NA,IR1,Lvl,AllPub,...,0,NA,NA,NA,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NA,IR1,Lvl,AllPub,...,0,NA,NA,NA,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NA,IR1,Lvl,AllPub,...,0,NA,NA,NA,0,12,2008,WD,Normal,250000


In [5]:
display(df.tail())

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
1455,1456,60,RL,62.0,7917,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,8,2007,WD,Normal,175000
1456,1457,20,RL,85.0,13175,Pave,NA,Reg,Lvl,AllPub,...,0,NA,MnPrv,NA,0,2,2010,WD,Normal,210000
1457,1458,70,RL,66.0,9042,Pave,NA,Reg,Lvl,AllPub,...,0,NA,GdPrv,Shed,2500,5,2010,WD,Normal,266500
1458,1459,20,RL,68.0,9717,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,4,2010,WD,Normal,142125
1459,1460,20,RL,75.0,9937,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,6,2008,WD,Normal,147500


In [6]:
display(
    df.sample(
        n=5,
        random_state=42,
    )
)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
892,893,20,RL,70.0,8414,Pave,NA,Reg,Lvl,AllPub,...,0,NA,MnPrv,NA,0,2,2006,WD,Normal,154500
1105,1106,60,RL,98.0,12256,Pave,NA,IR1,Lvl,AllPub,...,0,NA,NA,NA,0,4,2010,WD,Normal,325000
413,414,30,RM,56.0,8960,Pave,Grvl,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,3,2010,WD,Normal,115000
522,523,50,RM,50.0,5000,Pave,NA,Reg,Lvl,AllPub,...,0,NA,NA,NA,0,10,2006,WD,Normal,159000
1036,1037,20,RL,89.0,12898,Pave,NA,IR1,HLS,AllPub,...,0,NA,NA,NA,0,9,2009,WD,Normal,315500


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          1460 non-null   object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

### Типы данных

После контролируемой загрузки:

- 43 столбца имеют тип `object`;
- 35 столбцов имеют тип `int64`;
- 3 столбца имеют тип `float64`.

Тип pandas не всегда совпадает с фактической природой признака.

Например:

- `MSSubClass` хранится как число, но числа являются кодами типов домов;
- `OverallQual` и `OverallCond` являются порядковыми оценками от 1 до 10;
- `MoSold` является номером месяца;
- `YrSold`, `YearBuilt`, `YearRemodAdd` и `GarageYrBlt` являются временными признаками;
- `Id` является идентификатором, а не характеристикой дома.

## 4. Каталог всех признаков - собирает всю основную информацию о каждом столбце в одной таблице (справочник по датасету).

Для каждого признака мы одновременно видим:
- как называется столбец;
- что он означает;
- какой у него тип данных;
- сколько в нём уникальных значений;
- сколько пропусков;
- какие значения реально встречаются;
- является ли он признаком, идентификатором или таргетом.

Названия признаков сокращённые, и по ним не всегда понятно содержание:

Следующая ячейка извлекает краткие описания непосредственно из data_description.txt.
В текстовом описании используются названия Bedroom и Kitchen, тогда как в CSV соответствующие столбцы называются BedroomAbvGr и KitchenAbvGr.

In [9]:
description_text = DESCRIPTION_PATH.read_text(
    encoding="utf-8",
)

description_records = []

for line in description_text.splitlines():
    # Названия признаков находятся в строках без отступа
    # и отделяются от описания двоеточием.
    if (
        line
        and not line[0].isspace()
        and ":" in line
    ):
        feature, description = line.split(
            ":",
            maxsplit=1,
        )

        description_records.append(
            {
                "feature": feature.strip(),
                "description": description.strip(),
            }
        )


description_df = pd.DataFrame(
    description_records
)


# Названия в data_description.txt отличаются
# от названий столбцов в CSV.
description_df["feature"] = (
    description_df["feature"].replace(
        {
            "Bedroom": "BedroomAbvGr",
            "Kitchen": "KitchenAbvGr",
        }
    )
)


additional_descriptions = pd.DataFrame(
    [
        {
            "feature": "Id",
            "description": (
                "Unique identifier of the property."
            ),
        },
        {
            "feature": "SalePrice",
            "description": (
                "Property sale price in dollars. "
                "Target variable."
            ),
        },
    ]
)

description_df = pd.concat(
    [
        additional_descriptions,
        description_df,
    ],
    ignore_index=True,
)

In [10]:
def get_example_values(
    series: pd.Series,
    n_values: int = 5,
) -> list:
    """
    Возвращает несколько примеров значений столбца.
    """
    return (
        series
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(n_values)
        .tolist()
    )


feature_catalog = pd.DataFrame(
    {
        "feature": df.columns,
        "role": [
            (
                "identifier"
                if column == ID_COLUMN
                else "target"
                if column == TARGET
                else "feature"
            )
            for column in df.columns
        ],
        "dtype": df.dtypes.astype(str).values,
        "n_unique": [
            df[column].nunique(
                dropna=True
            )
            for column in df.columns
        ],
        "missing_count": [
            df[column].isna().sum()
            for column in df.columns
        ],
        "missing_pct": [
            df[column].isna().mean() * 100
            for column in df.columns
        ],
        "example_values": [
            get_example_values(
                df[column]
            )
            for column in df.columns
        ],
    }
)

feature_catalog = feature_catalog.merge(
    description_df,
    on="feature",
    how="left",
)

feature_catalog["missing_pct"] = (
    feature_catalog["missing_pct"].round(2)
)

pd.set_option(
    "display.max_colwidth",
    120,
)

display(feature_catalog)

,feature,role,dtype,n_unique,missing_count,missing_pct,example_values,description
0,Id,identifier,int64,1460,0,0.00,"[1, 2, 3, 4, 5]",Unique identifier of the property.
1,MSSubClass,feature,int64,15,0,0.00,"[60, 20, 70, 50, 190]",Identifies the type of dwelling involved in the sale.
2,MSZoning,feature,object,5,0,0.00,"[RL, RM, C (all), FV, RH]",Identifies the general zoning classification of the sale.
3,LotFrontage,feature,float64,110,259,17.74,"[65.0, 80.0, 68.0, 60.0, 84.0]",Linear feet of street connected to property
4,LotArea,feature,int64,1073,0,0.00,"[8450, 9600, 11250, 9550, 14260]",Lot size in square feet
...,...,...,...,...,...,...,...,...
76,MoSold,feature,int64,12,0,0.00,"[2, 5, 9, 12, 10]",Month Sold (MM)
77,YrSold,feature,int64,5,0,0.00,"[2008, 2007, 2006, 2009, 2010]",Year Sold (YYYY)
78,SaleType,feature,object,9,0,0.00,"[WD, New, COD, ConLD, ConLI]",Type of sale
79,SaleCondition,feature,object,6,0,0.00,"[Normal, Abnorml, Partial, AdjLand, Alloca]",Condition of sale
